# MotorPulse machine learning - from sensor data to a prediction

This notebook is written for readers who are new to machine learning. It walks through the process in small steps and explains what each step means.

We will use three sensor measurements as model inputs:

1. Vibration RMS acceleration (`acceleration_vector_rms_m_s2`)
2. Motor temperature (`temperature_c`)
3. Motor current (`current_amps`)

The model will try to predict `fault_status`.

> **Important: this is practice data, not real motor test data.** The CSV used here was mathematically generated. Its fault labels are simulated examples, not faults confirmed by inspecting a physical motor. Any accuracy reported here only describes this generated dataset. It must not be presented as real MotorPulse performance or used to make safety decisions.

## How to use this notebook

1. In VS Code, select a Python notebook kernel that has `numpy`, `pandas`, and `scikit-learn` installed.
2. Run the code cells from the top of the notebook to the bottom. In VS Code, **Shift+Enter** runs the selected cell and moves to the next one.
3. Read the text between code cells. It explains what the next code cell is doing and what its output means.

A code cell depends on values made by earlier cells. If you restart the notebook kernel, run the cells again from the beginning.

## Step 1 - Understand the prediction problem

Think of one CSV row as one short motor-sensor window: a small group of readings summarized as one example.

- **Inputs** are measurements the model can look at: vibration, temperature, and current.
- **Target** is the answer the model is learning to predict: `fault_status`.
- In this practice file, `fault_status = 0` means the generator called the window nominal, and `fault_status = 1` means the generator created an anomaly (an unusual simulated pattern). These are not verified real motor labels or fault diagnoses.

Machine learning finds patterns in examples with known answers, then uses those patterns to guess the answer for new examples. Here we will train on earlier rows and test on later rows.

## Step 2 - Load the Python tools and find the data file

These imported packages do the work. In the code, `pd` is the short name for pandas and `np` is the short name for NumPy:

- **pandas** reads and organizes the CSV as a table.
- **NumPy** helps check that measurements are valid numbers.
- **scikit-learn** provides the baseline and Random Forest models, plus evaluation measures.

The path-search code checks the current folder and its parent folders because VS Code may start the notebook from different locations. It prints the exact CSV path it found.

In [81]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

DATA_FILENAME = "motorpulse_scientific_synthetic_50000.csv"

# These are the only three measurements the model will receive.
FEATURES = [
    "acceleration_vector_rms_m_s2",
    "temperature_c",
    "current_amps",
]

# This is the answer column the model will learn to predict.
TARGET = "fault_status"

search_roots = [Path.cwd(), *Path.cwd().parents]
possible_paths = []
for folder in search_roots:
    possible_paths.append(folder / DATA_FILENAME)
    possible_paths.append(folder / "MotorPulsePoc" / DATA_FILENAME)

DATA_PATH = next((path.resolve() for path in possible_paths if path.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        f"Could not find {DATA_FILENAME}. Make sure it is in the MotorPulsePoc folder."
    )

print(f"CSV file found at: {DATA_PATH}")

CSV file found at: D:\GTA\ai-project\MotorPulsePoc\motorpulse_scientific_synthetic_50000.csv


If the cell prints a CSV path, continue. If it raises `FileNotFoundError`, the notebook could not find the file; check that `motorpulse_scientific_synthetic_50000.csv` is in the `MotorPulsePoc` folder.

## Step 3 - Read the CSV and look at a few rows

`pd.read_csv` turns the file into a table named `data`. `head()` displays the first five rows so you can see the columns and example values. The code prints the number of rows and columns. Columns such as `window_id`, `recorded_at_utc`, and `motor_id` help identify and order records; the model will not use them as sensor inputs.

In [82]:
data = pd.read_csv(DATA_PATH)

print(f"Number of rows: {len(data):,}")
print(f"Number of columns: {len(data.columns)}")
print("Column names:")
print(data.columns.tolist())

display(data.head())

Number of rows: 50,000
Number of columns: 10
Column names:
['window_id', 'recorded_at_utc', 'motor_id', 'acceleration_vector_rms_m_s2', 'temperature_c', 'current_amps', 'fault_status', 'fault_type', 'label_notes', 'data_source']


,window_id,recorded_at_utc,motor_id,acceleration_vector_rms_m_s2,temperature_c,current_amps,fault_status,fault_type,label_notes,data_source
0,SIM-00001,2026-01-01T00:00:00Z,SIM-MOTOR-01,0.396742,24.997,1.426513,0,synthetic_nominal,Mathematical simulation; not a physical MotorP...,synthetic_physics_inspired_v1
1,SIM-00002,2026-01-01T00:00:01.280000Z,SIM-MOTOR-01,0.623727,25.011,1.454236,0,synthetic_nominal,Mathematical simulation; not a physical MotorP...,synthetic_physics_inspired_v1
2,SIM-00003,2026-01-01T00:00:02.560000Z,SIM-MOTOR-01,0.648082,25.025,1.460487,0,synthetic_nominal,Mathematical simulation; not a physical MotorP...,synthetic_physics_inspired_v1
3,SIM-00004,2026-01-01T00:00:03.840000Z,SIM-MOTOR-01,0.477845,25.046,1.427068,0,synthetic_nominal,Mathematical simulation; not a physical MotorP...,synthetic_physics_inspired_v1
4,SIM-00005,2026-01-01T00:00:05.120000Z,SIM-MOTOR-01,0.597630,25.050,1.469490,0,synthetic_nominal,Mathematical simulation; not a physical MotorP...,synthetic_physics_inspired_v1


## Step 4 - Check the measurements and labels

Before training, check that the expected columns exist and that sensor readings are usable numbers. The timestamp is used to put readings in time order; it is **not** given to the model as an input.

The label-count table tells us how many rows are marked `0` (normal) and `1` (simulated anomaly). Checking these counts matters because a model can look accurate just by guessing the more common label.

## How to read the sensor summary table

Each **row** is one sensor. Each **column** describes that sensor's 50,000 readings. The values use the unit shown in the sensor name: vibration is in meters per second squared (`m/s^2`), temperature is in degrees Celsius (`C`), and current is in amperes (`A`).

- **count** means how many readings were included. Here every sensor has `50,000`, the same as the number of dataset rows. This tells us there are no missing values in these three columns; it does not prove that the readings came from a real motor.
- **mean** is the average across all readings. For example, the average temperature is about `35.32 C`.
- **50%** is the median, or middle reading after sorting from lowest to highest. Half the readings are at or below it and half are at or above it. For temperature, the median is `35.146 C`, close to the `35.324 C` mean, so the average and middle reading are similar in this dataset.
- **std** (standard deviation) describes how much readings tend to vary around their mean. For example, the temperature standard deviation is about `3.90 C`. It is not a sensor error and not a safe operating limit. Compare this value only with the same sensor measured in the same unit and conditions; do not compare vibration's `0.293` to temperature's `3.899` as if temperature varies more, because their units differ.
- **25% and 75%** mark the lower and upper edges of the middle half of readings. For vibration RMS, the middle half is between `0.374` and `0.651 m/s^2`. For temperature, it is between `32.278` and `38.210 C`. For current, it is between `0.936` and `1.728 A`.
- **min** and **max** are the smallest and largest values found in this file. For example, the recorded temperature values range from about `25.00 C` to `47.30 C`. These are only the extremes in this dataset, **not** recommended or safe motor limits.

### A simple way to check the table

1. Check that each `count` matches the number of rows. A smaller count can mean missing readings.
2. Look at the `25%`, `50%`, and `75%` values to see where most readings fall; one extreme `min` or `max` may be unusual.
3. Compare `mean` and `50%`. If they are far apart, a few very high or low readings may be pulling the average away from the typical middle reading.
4. Check unusual values against the sensor documentation and the motor's known speed and load. This table alone cannot tell whether a motor is healthy or faulty.

**Important:** this CSV contains synthetic (computer-generated) practice data and simulated labels, not measurements confirmed from a physical motor. The numbers explain how to read the table; they are not real motor operating limits or evidence of real-world model performance.

In [83]:
required_columns = FEATURES + [TARGET, "recorded_at_utc"]
missing_columns = [name for name in required_columns if name not in data.columns]
if missing_columns:
    raise ValueError(f"CSV is missing required columns: {missing_columns}")
if data.empty:
    raise ValueError("The CSV has no data rows.")

# Convert input and target columns to numbers; invalid text becomes a missing value.
for name in FEATURES + [TARGET]:
    data[name] = pd.to_numeric(data[name], errors="coerce")

if data[FEATURES + [TARGET]].isna().any().any():
    raise ValueError("A sensor value or target is missing or is not numeric.")
if not np.isfinite(data[FEATURES + [TARGET]].to_numpy(dtype=float)).all():
    raise ValueError("Sensor values and target must be finite numbers.")
if not set(data[TARGET].unique()).issubset({0, 1}):
    raise ValueError("fault_status must contain only 0 or 1.")

# ISO8601 handles timestamps with and without fractional seconds.
data["recorded_at_utc"] = pd.to_datetime(
    data["recorded_at_utc"], utc=True, format="ISO8601", errors="raise"
)
data = data.sort_values("recorded_at_utc", kind="stable").reset_index(drop=True)
data[TARGET] = data[TARGET].astype("int8")

print("Rows in each target class (simulated labels):")
display(data[TARGET].value_counts().sort_index().rename("number of rows").to_frame())

print("Typical ranges in the three sensor inputs:")
display(data[FEATURES].describe().T)

Rows in each target class (simulated labels):


,number of rows
fault_status,
0,37486
1,12514


Typical ranges in the three sensor inputs:


,count,mean,std,min,25%,50%,75%,max
acceleration_vector_rms_m_s2,50000.0,0.541429,0.292957,0.040000,0.374266,0.522045,0.650699,2.974945
temperature_c,50000.0,35.323794,3.898670,24.997000,32.278000,35.146000,38.210000,47.301000
current_amps,50000.0,1.349486,0.482566,0.303801,0.936093,1.386033,1.727291,3.527720


## Optional check - compare averages by label

This table compares each sensor's average for label `0` and label `1`. A difference can show a pattern the model may learn. Similar averages do not prove a sensor is useless, and different averages do not prove the sensor caused an anomaly. Here the labels and sensor values are synthetic, so use this only to explore the generated examples.

In [84]:
average_by_label = data.groupby(TARGET)[FEATURES].mean().round(3)
average_by_label.index = average_by_label.index.map(
    {0: "Nominal (0)", 1: "Simulated anomaly (1)"}
)
display(average_by_label)

,acceleration_vector_rms_m_s2,temperature_c,current_amps
fault_status,,,
Nominal (0),0.512,34.778,1.332
Simulated anomaly (1),0.630,36.960,1.402


## Step 5 - Separate the inputs from the answer

The model should see only the three sensor measurements. The ID, time, notes, and fault description are not used as inputs.

- `X` is the table of sensor inputs.
- `y` is the list of target answers.

Keeping these separate helps prevent the model from accidentally learning from information that would not be available as a sensor measurement.

In [85]:
X = data[FEATURES]
y = data[TARGET]

print("Model input columns:", X.columns.tolist())
print("Target column:", TARGET)
print("Number of examples:", len(X))

Model input columns: ['acceleration_vector_rms_m_s2', 'temperature_c', 'current_amps']
Target column: fault_status
Number of examples: 50000


## Step 6 - Set aside later rows for a fair test

We use the first 80% of the timeline to teach the model and keep the last 20% for the final test. That way, the test examples come later in time than the training examples, similar to using past data to predict future readings.

`X_train` and `y_train` are for learning. `X_test` and `y_test` are held back until evaluation. The timestamp controls the order only; it is not included in the sensor inputs.

The printed row counts show how many examples are in each time period. The label-count tables show how many normal (`0`) and anomaly (`1`) examples each period contains. The check stops if either period is missing a class.

In [86]:
split_at = int(len(data) * 0.80)
if split_at == 0 or split_at == len(data):
    raise ValueError("Not enough rows to create training and test sets.")

X_train = X.iloc[:split_at]
y_train = y.iloc[:split_at]
X_test = X.iloc[split_at:]
y_test = y.iloc[split_at:]

if y_train.nunique() != 2 or y_test.nunique() != 2:
    raise ValueError(
        "The training and test periods must each contain both labels, 0 and 1."
    )

print(f"Training examples: {len(X_train):,}")
print(f"Test examples: {len(X_test):,}")
print("Training label counts:")
display(y_train.value_counts().sort_index().rename("number of rows").to_frame())
print("Test label counts:")
display(y_test.value_counts().sort_index().rename("number of rows").to_frame())

Training examples: 40,000
Test examples: 10,000
Training label counts:


,number of rows
fault_status,
0,29800
1,10200


Test label counts:


,number of rows
fault_status,
0,7686
1,2314


## Step 7 - Create a simple baseline

A **baseline** is the simplest result we use as a yardstick. Before asking whether the Random Forest is useful, we compare it with an intentionally simple guess. Scikit-learn's `DummyClassifier` is a tool for making this kind of basic reference prediction; it is not the motor-fault model we want to use.

### What this baseline does

Imagine answering every question on a quiz with the answer that appeared most often before. That is what `strategy="most_frequent"` does: it counts the labels in the training answers (`y_train`), picks the most common one, then gives that same answer for every test row. It does **not** look at vibration, temperature, or current to make a decision.

In our training data, there are `29,800` normal (`0`) rows and `10,200` simulated anomaly (`1`) rows. Therefore, the baseline picks `0` and predicts normal for all `10,000` test rows. The code still passes `X_train` and `X_test` to the classifier because scikit-learn uses a common interface for models; this `most_frequent` strategy ignores those sensor inputs.

### What each code line does

- `DummyClassifier(strategy="most_frequent")` creates the simple majority-guessing classifier.
- `baseline.fit(X_train, y_train)` tells it to learn from the training set. For this strategy, it only needs the training labels to find the most common answer.
- `baseline.predict(X_test)` repeats that answer for every held-back test row.
- `accuracy_score(...)` counts the fraction of test guesses that match the known test labels.

The output shows `76.9%` accuracy because `7,686` of the `10,000` test rows are normal, and the baseline guessed normal every time. But it detected **zero** anomalies. This is why accuracy by itself can be misleading: a simple guess can score well when one label is more common. The Random Forest should be compared with this yardstick, especially on how many simulated anomalies it finds (fault recall). All these labels are synthetic, so this is a teaching example, not a real fault detector.

In [87]:
# Create a deliberately simple reference: always guess the most common training label.
baseline = DummyClassifier(strategy="most_frequent")

# Learn which label is most common. This strategy ignores the sensor measurements.
baseline.fit(X_train, y_train)

# Predict that same label for every held-back test example.
baseline_predictions = baseline.predict(X_test)

# Compare guesses with the known test labels and calculate the fraction correct.
baseline_accuracy = accuracy_score(y_test, baseline_predictions)

print(f"Baseline accuracy: {baseline_accuracy:.1%}")

Baseline accuracy: 76.9%


## Step 8 - Train the Random Forest

### What is a Random Forest?

A Random Forest is a group of decision trees working together. A decision tree is like a flowchart of questions. It might learn questions such as “Is temperature above a certain value?” and then “Is vibration above another value?” The computer learns useful questions from the training examples; we do not type those limits into this code.

Each tree learns from a slightly different random sample of the training rows and considers different sensor choices while building its questions. For a new row, the trees each make a prediction, and the forest combines their results to choose the class with the strongest support. Using many trees can make the result less dependent on the quirks of one tree—like asking a group instead of relying on one person's opinion.

### What happens in this code?

`RandomForestClassifier(...)` sets up the model. `model.fit(X_train, y_train)` is the learning step: it shows the model the three sensor inputs (`X_train`) alongside their known practice labels (`y_train`). It does **not** show it the held-back test rows. After fitting, `model` contains the learned trees and can predict `0` (nominal) or `1` (simulated anomaly) from sensor readings.

The settings mean:

- `n_estimators=300`: build 300 decision trees. More trees can make predictions more stable, but also take longer to train.
- `class_weight="balanced_subsample"`: give extra learning weight to the less common label in each tree's training sample. This helps the model pay attention to simulated anomalies instead of mostly learning the more common normal label. It does not add new anomaly examples or guarantee that anomalies will be found.
- `random_state=42`: use a fixed starting point for the model's random choices, so repeated runs with the same data and software can give repeatable results. The number `42` has no special ML meaning.
- `n_jobs=-1`: let scikit-learn use available processor cores to build trees faster; this affects speed, not the intended prediction method.

A Random Forest generally does not need feature normalization. It learns questions such as whether a value is above a threshold, rather than comparing distances between temperature, current, and vibration. The printed message `Training finished` only means the learning step completed; it does not mean the model is accurate. The next steps test its predictions. Because this notebook uses generated readings and labels, the trees learn patterns in this synthetic practice data—not confirmed rules for real motor faults.

In [88]:
model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1,
)

# Learn the relationship between sensor inputs and training labels.
model.fit(X_train, y_train)
print("Training finished.")

Training finished.


## Step 9 - Ask the model to predict the test rows

Now the trained model predicts labels for the held-back sensor readings. We compare these guesses with the known test labels. The test data is for checking the model; it is not used to train it.

In the preview table, `actual_fault_status` is the known simulated answer, `predicted_fault_status` is the model's guess, and `predicted_anomaly_score` is the model's score for class `1` (simulated anomaly). A high score is not a confirmed real fault.

In [89]:
predictions = model.predict(X_test)
fault_scores = model.predict_proba(X_test)[:, 1]

print("First 10 comparisons: actual label versus model prediction")
display(pd.DataFrame({
    "actual_fault_status": y_test.iloc[:10].to_numpy(),
    "predicted_fault_status": predictions[:10],
    "predicted_anomaly_score": fault_scores[:10].round(3),
}))

First 10 comparisons: actual label versus model prediction


,actual_fault_status,predicted_fault_status,predicted_anomaly_score
0,0,0,0.127
1,0,0,0.210
2,0,0,0.247
3,0,0,0.087
4,0,0,0.020
5,0,0,0.297
6,0,0,0.167
7,0,0,0.077
8,0,0,0.033
9,0,0,0.100


## Step 10 - Measure performance

Accuracy answers: "What fraction of all test examples did the model label correctly?" Accuracy alone can hide missed faults, especially when one label is more common than the other. We also calculate:

- **Precision for faults:** Of the windows predicted as faults, how many were labeled faults?
- **Recall for faults:** Of the windows labeled as faults, how many did the model find? Low recall means missed faults.
- **F1 for faults:** A single score that balances precision and recall.
- **ROC-AUC:** How well the model's scores rank fault examples above normal examples. A value near 0.5 is like a random ranking; closer to 1 is better.

These scores describe only the simulated data in this notebook. In the detailed report below, `support` is the number of test examples for that class; `macro avg` gives both classes equal weight, while `weighted avg` gives more weight to the class with more examples.

In [90]:
model_accuracy = accuracy_score(y_test, predictions)
fault_precision = precision_score(y_test, predictions, pos_label=1, zero_division=0)
fault_recall = recall_score(y_test, predictions, pos_label=1, zero_division=0)
fault_f1 = f1_score(y_test, predictions, pos_label=1, zero_division=0)
fault_roc_auc = roc_auc_score(y_test, fault_scores)

print(f"Random Forest accuracy: {model_accuracy:.1%}")
print(f"Fault precision:        {fault_precision:.1%}")
print(f"Fault recall:           {fault_recall:.1%}")
print(f"Fault F1 score:         {fault_f1:.1%}")
print(f"ROC-AUC:                {fault_roc_auc:.3f}")
print(f"Majority baseline accuracy: {baseline_accuracy:.1%}")

Random Forest accuracy: 80.3%
Fault precision:        59.4%
Fault recall:           46.8%
Fault F1 score:         52.4%
ROC-AUC:                0.768
Majority baseline accuracy: 76.9%


## Step 11 - See what kinds of mistakes it made

A confusion matrix counts correct and incorrect guesses. Rows are the actual label; columns are the model's prediction:

- **True normal:** actual `0`, predicted `0`.
- **Missed anomaly:** actual `1`, predicted `0`.
- **False alarm:** actual `0`, predicted `1`.
- **Detected anomaly:** actual `1`, predicted `1`.

To read the counts, start with the row for the actual label. In the `Actual anomaly (1)` row, the predicted-anomaly column shows anomalies found, and the predicted-normal column shows anomalies missed. In this run, the model found `1,084` and missed `1,230` of the `2,314` simulated anomalies, which gives the `46.8%` fault recall shown above. In the `Actual normal (0)` row, `742` entries were false alarms. These are counts from this synthetic test set, not real motor faults.

For maintenance, missed faults and false alarms can have different costs. This synthetic exercise does not set operational alarm thresholds.

In [91]:
counts = confusion_matrix(y_test, predictions, labels=[0, 1])
confusion_table = pd.DataFrame(
    counts,
    index=["Actual normal (0)", "Actual anomaly (1)"],
    columns=["Predicted normal (0)", "Predicted anomaly (1)"],
)
display(confusion_table)

print("Detailed scores for each label:")
print(classification_report(
    y_test,
    predictions,
    labels=[0, 1],
    target_names=["normal (0)", "anomaly (1)"],
    zero_division=0,
))

,Predicted normal (0),Predicted anomaly (1)
Actual normal (0),6944,742
Actual anomaly (1),1230,1084


Detailed scores for each label:
              precision    recall  f1-score   support

  normal (0)       0.85      0.90      0.88      7686
 anomaly (1)       0.59      0.47      0.52      2314

    accuracy                           0.80     10000
   macro avg       0.72      0.69      0.70     10000
weighted avg       0.79      0.80      0.79     10000



## Step 12 - See which measurements the forest used

This Random Forest can report its own built-in feature importance. The values estimate how useful each sensor input was in the forest's tree splits. They are relative scores that add up to about 1; for example, `0.45` is about 45% of this model's total split-importance score. It is not a 45% chance of fault or proof that a sensor caused an anomaly. In this run, temperature's score is largest, but that only tells us it was used most by this forest on this synthetic data. These values may mainly reflect how the data was generated.

In [92]:
importance = pd.Series(
    model.feature_importances_,
    index=FEATURES,
    name="relative importance",
).sort_values(ascending=False)
display(importance.to_frame())

,relative importance
temperature_c,0.446315
acceleration_vector_rms_m_s2,0.277292
current_amps,0.276392


## Step 13 - General feature check: permutation importance

The previous importance values are specific to Random Forest. **Permutation importance** is a general method that can be used with many models. It asks: "If I scramble one sensor column, how much worse do the model's predictions get?"

The code shuffles one column in the held-back test data and measures the change in fault F1 score. A large positive `mean_f1_drop` means the model relied more on that input for this test set. For example, `0.10` means the F1 score fell by about 0.10 points (10 percentage points) after shuffling. A value near zero means shuffling it changed the score very little. A negative value means the score happened to improve after shuffling. `variation` is the standard deviation: a rough measure of how spread out the 10 shuffle results were.

These values are **score changes**, not percentages or probabilities. This method describes how this model uses these inputs; it does not prove that a sensor causes faults. Related sensors can share importance. Because this notebook uses generated data, the result is only a practice example. If you use this result to change the model or choose inputs, reserve a new untouched time period for the final evaluation.

In [93]:
from sklearn.inspection import permutation_importance

# F1 focuses on finding the simulated fault class (label 1).
permutation_result = permutation_importance(
    model,
    X_test,
    y_test,
    scoring="f1",
    n_repeats=10,
    random_state=42,
    n_jobs=-1,
)

permutation_table = pd.DataFrame(
    {
        "mean_f1_drop": permutation_result.importances_mean,
        "variation": permutation_result.importances_std,
    },
    index=FEATURES,
).sort_values("mean_f1_drop", ascending=False)

print("Feature importance as change in fault F1 after shuffling:")
display(permutation_table)

Feature importance as change in fault F1 after shuffling:


,mean_f1_drop,variation
temperature_c,0.208144,0.004351
acceleration_vector_rms_m_s2,0.112664,0.004509
current_amps,0.091122,0.003823


### How to understand this table

The model's original fault F1 score is about `0.524` (from Step 10). `mean_f1_drop` is how much that score fell, on average, after scrambling one sensor's values. Subtract the drop from `0.524` to estimate the score after scrambling:

| Sensor shuffled | Average F1 drop | Approx. F1 after shuffle | Plain-language reading |
| --- | ---: | ---: | --- |
| Temperature | `0.208` | `0.316` | Biggest drop; this model depended most on temperature in this test. |
| Vibration RMS | `0.113` | `0.411` | The model also used vibration information. |
| Current | `0.091` | `0.433` | The model used current too, with a smaller measured effect. |

A drop of `0.208` is about 20.8 **F1 percentage points**; it does not mean temperature caused 20.8% of faults or that there is a 20.8% fault probability. `variation` is the standard deviation across the 10 shuffles. Values around `0.004` mean those shuffles gave similar drops on this same test set; they do not prove the result will hold on another motor or real data.

**Bottom line:** temperature ranked highest for this trained model on this synthetic test set. This is a description of the model's behavior, not a physical diagnosis or a claim that temperature is the most important real-world sensor.

## Step 14 - Try one example

The next cell shows the three measurements for one test window, the model's prediction, and the known simulated label. The leftmost number in the sensor table is its row index, not another sensor measurement. The anomaly score is the model's score for class `1`; it is not a guaranteed probability that a real motor has a fault. `predict` uses scikit-learn's default decision rule to choose a class; this notebook does not set a real alarm threshold.

In [94]:
example_number = 0
example_inputs = X_test.iloc[[example_number]]
example_prediction = int(model.predict(example_inputs)[0])
example_score = float(model.predict_proba(example_inputs)[0, 1])
example_actual = int(y_test.iloc[example_number])

print("Sensor measurements:")
display(example_inputs)
print(f"Actual simulated label: {example_actual}")
print(f"Predicted label: {example_prediction}")
print(f"Model anomaly score: {example_score:.1%}")

Sensor measurements:


,acceleration_vector_rms_m_s2,temperature_c,current_amps
40000,0.736568,34.764,1.894831


Actual simulated label: 0
Predicted label: 0
Model anomaly score: 12.7%


### How to interpret this single example

Compare the actual label with the predicted label: if they match, this one prediction is correct. In the displayed example both are `0`, so that row was classified correctly. The `12.7%` anomaly score describes only this row; it is **not** the model's accuracy.

Overall accuracy, shown earlier, counts correct predictions across all 10,000 test rows. Fault recall answers a different question: what fraction of the simulated anomaly rows (`1`) did the model find? One correct example does not guarantee that the model performs well overall. All these results use synthetic data.

## Quick glossary

- **Feature / input:** a sensor measurement given to the model.
- **RMS (root mean square):** a way to summarize changing vibration readings as one overall vibration level for a short measurement window.
- **Target / label:** the example answer the model learns to predict.
- **Training data:** earlier examples used to teach the model.
- **Test data:** later examples held aside to check the model on examples it did not train on.
- **Baseline:** a simple reference guess that the trained model should improve upon.
- **Fit:** learn patterns from the training examples.
- **Predict:** use learned patterns to guess labels for new examples.
- **Accuracy:** fraction of all guesses that are correct.
- **Precision:** when the model flags an anomaly, how often the label is actually 1.
- **Recall:** of all examples labeled 1, how many the model finds.
- **F1:** one score that balances precision and recall.
- **ROC-AUC:** how well model scores rank examples labeled 1 above examples labeled 0; about 0.5 is a random ranking.
- **Confusion matrix:** a count of correct guesses and mistakes, grouped by actual and predicted labels.
- **Anomaly score:** the model's score for the anomaly class; it is not a verified fault diagnosis.
- **Feature importance:** an estimate of how useful an input was to a particular model; it does not show cause and effect.
- **Permutation importance:** shuffle one input and measure how much a chosen score changes.
- **Standard deviation (`std`):** a measure of how spread out values are around their average.
- **Normalization:** rescaling measurements to a common range. This notebook does not normalize inputs; a Random Forest generally does not require it.

## Final summary - what this experiment does and does not show

- The model takes **three inputs only**: vibration RMS, temperature, and current. It predicts the separate target `fault_status`.
- The notebook compares a simple majority guess with a Random Forest and evaluates both on later rows.
- Check fault recall and the confusion matrix as well as accuracy; accuracy by itself can be misleading.
- The dataset and its labels are synthetic. A good score here does **not** prove the model can detect real motor faults.
- For a real evaluation, collect physical MotorPulse readings, get condition labels verified, and test on a separate capture period or motor. Do not induce unsafe faults to obtain training examples.
- This is an educational three-input classifier. The existing MotorPulse Edge AI API uses a different feature set and a healthy-baseline detector, so this notebook is not a drop-in replacement for that API.